# 4. Agent with Tools — The ReAct Loop

## From pipeline to agent

Notebooks 1–3 had **you** deciding the flow. An **agent** flips this: the **LLM decides**
what to do next. You give it tools; it picks which to call, with what arguments,
and when it has enough information to answer.

The shape is a **cycle**:

```
START → agent ──(tool calls?)──→ tools ──→ agent → ... → END
            └──(no tool calls)──→ END
```

Three pieces make it work:

- `@tool` — decorator that turns a Python function into something the LLM can call
  (the docstring becomes the tool's description — write it well!)
- `llm.bind_tools(tools)` — tells the LLM which tools exist
- `ToolNode` + `tools_condition` — prebuilt node that executes tool calls, and a
  prebuilt router that checks "did the LLM ask for a tool?"

## Real-life example: personal trip assistant

"What's the weather in Tokyo, and how much is 250 USD in JPY at 155 yen/dollar?"
No single prompt can answer this reliably — it needs a weather lookup AND math.
The agent will call **two different tools**, then combine the results.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# .env in this repo stores the key as GOOGLE_API_KEY_1 — normalize it
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_API_KEY_1")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
llm.invoke("Say 'ready' if you can hear me.").content

In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # Real life: call OpenWeatherMap etc. Mocked here so the notebook always runs.
    fake_db = {
        "tokyo": "Sunny, 22°C, light breeze",
        "london": "Rainy, 11°C",
        "delhi": "Hazy, 34°C",
    }
    return fake_db.get(city.lower(), f"No data for {city}, assume mild and cloudy.")


@tool
def calculator(expression: str) -> str:
    """Evaluate a basic math expression, e.g. '250 * 155' or '84.5 * 0.15'."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Error: only basic arithmetic is allowed."
    try:
        return str(eval(expression))   # safe-ish: charset whitelisted above
    except Exception as e:
        return f"Error: {e}"


tools = [get_weather, calculator]

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

llm_with_tools = llm.bind_tools(tools)


def agent(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")
# tools_condition: routes to "tools" if the last AI message contains tool calls, else END
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")   # after running tools, go back and let the LLM think again

app = builder.compile()

In [ ]:
# Visualize the graph (needs internet for mermaid rendering — safe to skip)
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Could not render image, here is the mermaid source instead:\n")
    print(app.get_graph().draw_mermaid())

In [ ]:
question = "What's the weather in Tokyo, and how much is 250 USD in JPY at 155 yen per dollar?"

result = app.invoke({"messages": [("user", question)]})

# Replay the agent's reasoning trail
for m in result["messages"]:
    if m.type == "ai" and getattr(m, "tool_calls", None):
        for tc in m.tool_calls:
            print(f"  AGENT wants tool -> {tc['name']}({tc['args']})")
    elif m.type == "tool":
        print(f"  TOOL returned    -> {m.content}")
    elif m.type == "ai":
        print(f"\nFINAL ANSWER:\n{m.content}")

In [ ]:
# A question needing NO tools — watch it skip straight to END (no tool lines printed)
result = app.invoke({"messages": [("user", "Say hello in Japanese.")]})
print(result["messages"][-1].content)

## The prebuilt shortcut

The loop above is so common that you rarely hand-build it. One-liner version
(same graph under the hood, plus a system prompt):

In [ ]:
from langchain.agents import create_agent

prebuilt = create_agent(llm, tools, system_prompt="You are a concise trip assistant.")
out = prebuilt.invoke({"messages": [("user", "Weather in London? And what's 18% tip on 64?")]})
print(out["messages"][-1].content)

## Key takeaways

- An agent = LLM + tools + **a cycle**. The `tools → agent` edge is what makes it loop
  until the LLM stops requesting tools.
- Tool **docstrings are prompts** — the LLM chooses tools by reading them.
- Validate tool inputs (see the calculator's charset whitelist) — the LLM writes the
  arguments, and you should never blindly trust them.
- Real-world versions of this notebook: customer-data lookup bots, SQL agents,
  code execution assistants, browsing/research agents.

**Next:** notebook 5 — what if a tool does something risky, like sending an email?
Put a human in the loop.